# GraphSAGE edge-ablation: results analysis
This notebook **analyzes recorded experimental CSV files**. It does not train models or re-run the original research. Original authorship and methodology are described in the repository README and METHODOLOGY.md.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd()
ROOT = HERE if (HERE / 'results').exists() else HERE.parent
RESULTS = ROOT / 'results'
assert RESULTS.exists(), 'Open the notebook from the repository root or notebooks/ folder'
print('Reading results from', RESULTS.resolve())

## 1. Check dataset completeness
Each sampling scheme includes 5 seeds × 3 edge-removal settings. The training/validation/test split seed was held fixed at 4444.

In [ ]:
original = pd.read_csv(RESULTS / 'edge_ablation_original_five_seeds.csv')
unique = pd.read_csv(RESULTS / 'edge_ablation_unique_five_seeds.csv')
for label, df in [('Original', original), ('Unique-edge', unique)]:
    assert len(df) == 15
    assert df.groupby(['seed', 'training_edge_drop_requested']).size().eq(1).all()
    print(label, sorted(df.seed.unique()), sorted(df.training_edge_drop_requested.unique()))

## 2. Compute across-seed mean and sample SD
These values describe variability over five training seeds sharing a single data split, **not** independent experimental datasets.

In [ ]:
def summarize(df):
    return df.groupby('training_edge_drop_requested').agg(
        mean_test_mcc=('test_mcc', 'mean'),
        sd_test_mcc=('test_mcc', 'std'),
        mean_accuracy=('test_accuracy', 'mean')
    )

print('Original edge sampling:')
display(summarize(original))
print('Exact unique-edge sampling:')
display(summarize(unique))

## 3. Plot mean MCC with variability
Error bars indicate standard deviation across the five seeds.

In [ ]:
a, b = summarize(original), summarize(unique)
rates = [0.0, 0.2, 0.4]
x = np.arange(len(rates))
fig, ax = plt.subplots(figsize=(9, 5))
for displacement, summary, label in [(-0.18, a, 'Original sampling'), (0.18, b, 'Unique-edge sampling')]:
    summary = summary.reindex(rates)
    ax.bar(x + displacement, summary.mean_test_mcc, width=0.36,
           yerr=summary.sd_test_mcc, capsize=4, label=label)
ax.set_xticks(x, ['0%', '20%', '40%'])
ax.set_xlabel('Training-edge removal rate')
ax.set_ylabel('Mean test MCC ± SD')
ax.legend()
plt.show()

## 4. Paired per-seed differences relative to 0%
The pairings use identical seed labels. Interpreting differences as statistically significant would require further design and analysis.

In [ ]:
wide = unique.pivot(index='seed', columns='training_edge_drop_requested', values='test_mcc')
diffs = pd.DataFrame({'20% minus 0%':wide[0.2]-wide[0.0], '40% minus 0%':wide[0.4]-wide[0.0]})
display(diffs)
print('Mean paired differences:')
display(diffs.mean())

## Interpretation
Across five training seeds, corrected 40% edge removal has the highest mean test MCC, but only about 0.004 above the 0% baseline. The result is exploratory and does **not** demonstrate a reliable improvement. This pilot uses a small GraphSAGE model, 40 training epochs, and a single fixed split. It is not directly comparable with the larger original model benchmark.